# 03. panel

## 0. setup

In [1]:
import gc
from pathlib import Path
 
import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

## 1. config

In [2]:
in_treat = data_interim / 'treat_cartel.parquet'
in_clong = data_interim / 'cartel_long.parquet'
in_scope = data_interim / 'scope_long.parquet'
in_pat = data_interim / 'pat_isic3.parquet'
in_ipc_isic = data_raw / 'external' / 'ipc4_to_isic_rev4_3_excl_service.txt'    # ALP v2209 excl. services: industry universe
in_cartel = data_raw / 'cartel' / 'manual' / 'cartel_manual_v8.xlsx'            # ref_ctry_timeline

start   = 1978            # first priority year fully coverable at the EPO
ref_win = (2008, 2019)    # reference window for the truncation rule
k_sd    = 3               # cutoff: first year after ref_win below mean - k_sd * sd
k_pre   = 5               # minimum clean pre-years before onset
alp_artefact = []         # ISIC3 with implausible ALP mass; fill after 5.2, log the decision

out_panel = data_proc / 'panel.parquet'

c = ['isic3', 'ctry_iso']            # cell
k = c + ['year']                     # cell-year
ref  = pd.read_excel(in_cartel, sheet_name='ref_ctry_timeline')
isic = pd.read_csv(in_ipc_isic, dtype={'isic_rev4_3': str})['isic_rev4_3'].drop_duplicates()
print(f'universe: {len(isic)} isic3 x {len(ref)} countries | start {start} | ref {ref_win}, k_sd {k_sd}, k_pre {k_pre}')

universe: 110 isic3 x 31 countries | start 1978 | ref (2008, 2019), k_sd 3, k_pre 5


## universe

### 2.1 treatment histories

In [3]:
treat = pd.read_parquet(in_treat)
clong = pd.read_parquet(in_clong)

# episodes: spells of expo > 0; any zero year starts a new episode
t = treat.sort_values(k).copy()
t['ep'] = t.groupby(c)['year'].diff().ne(1).astype(int).groupby([t['isic3'], t['ctry_iso']]).cumsum()

# cell-level timing
hist = t.groupby(c).agg(g_on_any=('year', 'min'), n_ep=('ep', 'max'), treat_2d_cell=('treat_2d', 'min'))
hist['g_on']  = t[t['treated'] == 1].groupby(c)['year'].min()                        # main rule: first year expo >= 0.5
hist['g_off'] = t[t['ep'] == 1].groupby(c)['year'].max() + 1                         # first year after episode 1
hist['g_ep2'] = t[t['ep'] == 2].groupby(c)['year'].min()                             # start of episode 2
ep1 = clong.merge(t.loc[t['ep'] == 1, k], on=k)
hist['g_dec'] = ep1.groupby(c)['decision_year'].min()                                # earliest decision, episode 1
hist = hist.reset_index()

print(f'histories: {len(hist)} cells | multi-episode: {(hist["n_ep"] > 1).sum()} | '
      f'expo > 0 but never >= 0.5: {hist["g_on"].isna().sum()} | onset before {start}: {(hist["g_on_any"] < start).sum()}')

histories: 346 cells | multi-episode: 40 | expo > 0 but never >= 0.5: 5 | onset before 1978: 15


### 2.2 grid

In [4]:
pat = pd.read_parquet(in_pat)
y_max = int(pat['year'].max())

grid = (pd.DataFrame({'isic3': isic})
          .merge(ref[['ctry_iso']], how='cross')
          .merge(pd.DataFrame({'year': range(start, y_max + 1)}), how='cross'))

# documented drop: priority years before start
pre = pat['year'] < start
print(f'patents before {start} dropped: {pre.sum()} cell-years, pat_frac {pat.loc[pre, "pat_frac"].sum():.1f}')
pat = pat[~pre]
print(f'grid: {len(grid):,} cell-years = {len(isic)} x {len(ref)} x {y_max - start + 1} years ({start}-{y_max})')

patents before 1978 dropped: 787 cell-years, pat_frac 2041.6
grid: 163,680 cell-years = 110 x 31 x 48 years (1978-2025)


### 2.3 patents

In [5]:
assert pat['isic3'].isin(isic).all() and pat['ctry_iso'].isin(ref['ctry_iso']).all(), 'patent cells outside universe'
panel = grid.merge(pat, on=k, how='left')
panel[['pat_frac', 'cit3_frac']] = panel[['pat_frac', 'cit3_frac']].fillna(0)
assert np.isclose(panel['pat_frac'].sum(), pat['pat_frac'].sum()), 'mass lost'
print(f'patents: {len(pat):,} non-zero -> {len(panel):,} rows, {(panel["pat_frac"] == 0).mean():.1%} zero')

patents: 100,798 non-zero -> 163,680 rows, 38.4% zero


### 2.4 treatment

In [6]:
# documented drop: treated cells outside the ALP ex-services universe
out = ~treat['isic3'].isin(isic)
inf_all = clong['infringement_id'].nunique()
inf_in  = clong.loc[clong['isic3'].isin(isic), 'infringement_id'].nunique()
print(f'outside universe: {treat.loc[out, "isic3"].nunique()} isic3, {treat[out].groupby(c).ngroups} cells, '
      f'{treat.loc[out, "treated"].sum()} treated cell-years | infringements {inf_all} -> {inf_in}')

n0 = len(panel)
panel = panel.merge(treat.loc[~out, k + ['expo', 'treated', 'treat_2d']], on=k, how='left')
panel['expo']     = panel['expo'].fillna(0)
panel['treated']  = panel['treated'].fillna(0).astype(int)
panel['treat_2d'] = panel['treat_2d'].fillna(0).astype(int)
assert len(panel) == n0
print(f'treatment: {(panel["expo"] > 0).sum():,} cell-years expo > 0 (from {start}), {panel["treated"].sum():,} treated | '
      f'{panel.loc[panel["expo"] > 0].groupby(c).ngroups} cells')

outside universe: 13 isic3, 55 cells, 311 treated cell-years | infringements 153 -> 132
treatment: 3,181 cell-years expo > 0 (from 1978), 2,854 treated | 291 cells


## 3. sample window

### 3.1 end cutoffs (rule)

In [7]:
by_year = panel.groupby('year')[['pat_frac', 'cit3_frac']].sum()
series  = {'pat': by_year['pat_frac'], 'cit': by_year['cit3_frac'] / by_year['pat_frac']}

# last year = year before the first post-reference year below mean - k_sd * sd of the reference window
last = {}
for name, s in series.items():
    r   = s.loc[ref_win[0]:ref_win[1]]
    thr = r.mean() - k_sd * r.std()
    fail = s[(s.index > ref_win[1]) & (s < thr)]
    last[name] = int(fail.index.min() - 1) if len(fail) else int(s.index.max())
    print(f'{name}: ref mean {r.mean():.3f}, sd {r.std():.3f}, threshold {thr:.3f} | last year {last[name]} '
          f'(value {s.loc[last[name]]:.3f}), first fail {fail.index.min() if len(fail) else None}')

panel['in_pat'] = (panel['year'] <= last['pat']).astype(int)
panel['in_cit'] = (panel['year'] <= last['cit']).astype(int)

pat: ref mean 60563.485, sd 2217.792, threshold 53910.108 | last year 2022 (value 61201.338), first fail 2023
cit: ref mean 0.373, sd 0.018, threshold 0.320 | last year 2020 (value 0.371), first fail 2021


### 3.2 membership start

In [8]:
# country enters at max(start, first EU or EEA entry); exits not applied (GB: parking lot)
ref['t0'] = ref[['eu_entry_year', 'eea_entry_year']].min(axis=1).clip(lower=start).astype(int)
panel = panel.merge(ref[['ctry_iso', 't0']], on='ctry_iso')
panel['in_member'] = (panel['year'] >= panel['t0']).astype(int)
print(f'in_member: {panel["in_member"].mean():.1%} of rows | treated cell-years pre-membership: '
      f'{panel.loc[panel["in_member"] == 0, "treated"].sum()}')

in_member: 68.5% of rows | treated cell-years pre-membership: 135


## 4. treatment timing

### 4.1 cohorts

In [9]:
n0 = len(panel)
panel = panel.merge(hist, on=c, how='left')
assert len(panel) == n0
panel['n_ep'] = panel['n_ep'].fillna(0).astype(int)
print(f'cells with onset (expo > 0): {panel.loc[panel["g_on_any"].notna()].groupby(c).ngroups} | '
      f'with g_on: {panel.loc[panel["g_on"].notna()].groupby(c).ngroups}')

# did vars from g_on (never treated: post = 0, rel_time missing)
panel['cell_id']  = panel.groupby(c).ngroup()                         # integer unit id for estimators
panel['post']     = (panel['year'] >= panel['g_on']).astype(int)      # absorbing: 1 from onset on
panel['rel_time'] = panel['year'] - panel['g_on']                     # event time

print(f'cell_id: {panel["cell_id"].nunique():,} | post = 1: {panel["post"].sum():,} rows '
      f'| post = 1 & treated = 0 (after end or gaps): {((panel["post"] == 1) & (panel["treated"] == 0)).sum():,} '
      f'| rel_time {panel["rel_time"].min():.0f} to {panel["rel_time"].max():.0f}')

cells with onset (expo > 0): 291 | with g_on: 290
cell_id: 3,410 | post = 1: 8,927 rows | post = 1 & treated = 0 (after end or gaps): 6,073 | rel_time -39 to 56


### 4.2 early onset and repeat episodes

In [10]:
# onset relative to the cell's first sample year t0 (pre-periods must be free of any exposure: g_on_any)
panel['left_cens'] = (panel['g_on_any'] < panel['t0']).astype(int)                  # onset before first sample year
panel['few_pre']   = (panel['g_on_any'] - panel['t0'] < k_pre).astype(int)          # < k_pre clean pre-years (incl. left_cens)
panel['multi_ep']  = (panel['n_ep'] > 1).astype(int)
panel['post_ep2']  = (panel['year'] >= panel['g_ep2']).astype(int)                  # rows from episode 2 on

cl = panel.drop_duplicates(c)
print(f'cells: left_cens {cl["left_cens"].sum()} | few_pre {cl["few_pre"].sum()} | multi_ep {cl["multi_ep"].sum()} '
      f'| rows post_ep2 {panel["post_ep2"].sum():,}')

cells: left_cens 34 | few_pre 78 | multi_ep 37 | rows post_ep2 806


## 5. controls

### 5.1 exposure and scope

In [11]:
scope = pd.read_parquet(in_scope)
sc = scope[k].drop_duplicates().assign(in_scope=1)

n0 = len(panel)
panel = panel.merge(sc, on=k, how='left')
panel['in_scope'] = panel['in_scope'].fillna(0).astype(int)
assert len(panel) == n0

panel['never_exp']  = panel['g_on_any'].isna().astype(int)                                    # cell: no expo > 0 in any year
panel['scope_only'] = ((panel['in_scope'] == 1) & (panel['expo'] == 0)).astype(int)          # cell-year: in scope, no participant
panel['scope_ever'] = panel.groupby(c)['in_scope'].transform('max')                           # cell: ever in scope
panel['out_scope']  = ((panel['expo'] > 0) & (panel['in_scope'] == 0)).astype(int)           # cell-year: exposed, not in scope

# same isic3 exposed in another country that year
n_exp = panel.groupby(['isic3', 'year'])['expo'].transform(lambda x: (x > 0).sum())
panel['isic_exposed'] = ((n_exp - (panel['expo'] > 0)) > 0).astype(int)

print(f'scope_only cell-years: {panel["scope_only"].sum():,} | out_scope cell-years: {panel["out_scope"].sum():,} | '
      f'isic_exposed: {panel["isic_exposed"].mean():.1%} of rows')

scope_only cell-years: 8,622 | out_scope cell-years: 187 | isic_exposed: 12.5% of rows


### 5.2 patent support and ALP artefacts

In [12]:
s = (panel['in_pat'] == 1) & (panel['in_member'] == 1)

# cell with zero patents in every in-sample year
panel['pat_zero_cell'] = (panel['pat_frac'].where(s, 0).groupby([panel['isic3'], panel['ctry_iso']]).transform('sum') == 0).astype(int)

# mass by isic3: inspect before filling alp_artefact in config
share = panel[s].groupby('isic3')['pat_frac'].sum()
print((share / share.sum()).sort_values(ascending=False).head(8).round(4).to_string())
panel['alp_artefact'] = panel['isic3'].isin(alp_artefact).astype(int)
print(f'pat_zero_cell: {panel.loc[panel["pat_zero_cell"] == 1].groupby(c).ngroups} cells | alp_artefact: {alp_artefact}')

isic3
201    0.0783
210    0.0773
281    0.0771
263    0.0559
262    0.0527
360    0.0453
265    0.0352
267    0.0297
pat_zero_cell: 112 cells | alp_artefact: []


### 5.3 control counts by treated isic3

In [13]:
# cell types on in-sample rows (counts window, members); patent support required
cs = panel[s & (panel['pat_zero_cell'] == 0)].groupby(c).agg(treated=('treated', 'max'), exp=('expo', 'max'),
                                                            never_exp=('never_exp', 'max'), scope_ever=('scope_ever', 'max'))
cs['clean']      = (cs['never_exp'] == 1) & (cs['scope_ever'] == 0)
cs['scope_ctrl'] = (cs['never_exp'] == 1) & (cs['scope_ever'] == 1)
cs['exp_only']   = (cs['treated'] == 0) & (cs['exp'] > 0)                           # exposed in sample, never >= 0.5

tab = cs.groupby('isic3')[['treated', 'clean', 'scope_ctrl', 'exp_only']].sum().astype(int)
tab = tab[tab['treated'] > 0].sort_values('treated', ascending=False)
with pd.option_context('display.max_rows', None):
    print(tab.to_string())
print(f'treated isic3: {len(tab)} | treated cells {tab["treated"].sum()} | clean {tab["clean"].sum()} | '
      f'scope-only {tab["scope_ctrl"].sum()} | isic3 with 0 clean controls: {(tab["clean"] == 0).sum()}')

       treated  clean  scope_ctrl  exp_only
isic3                                      
201         18      0          12         0
222         17      1          13         0
241         13     13           5         0
293         13      1          17         0
202         13      3          15         0
203         12      3          16         0
281         12      1          18         0
332         11      2          16         0
310         11     20           0         0
242         10      1          20         0
192         10      3          18         0
239          9      1          21         0
251          9     18           2         0
271          8      1          22         0
383          6      0          25         0
170          6     13          12         0
291          6      0          25         0
273          6      1          24         0
243          6     12          11         0
432          5     19           0         0
231          5      3          2

## 6. save

In [14]:
flags = ['in_pat', 'in_cit', 'in_member', 'left_cens', 'few_pre', 'multi_ep', 'post_ep2', 'never_exp',
         'in_scope', 'scope_only', 'scope_ever', 'out_scope', 'isic_exposed', 'pat_zero_cell', 'alp_artefact', 'treat_2d']

# flag summary: rows, cells, treated cells (main rule) with flag = 1
tc = panel.groupby(c)['treated'].transform('max')
summ = pd.DataFrame({f: {'rows': panel[f].sum(),
                         'cells': panel.loc[panel[f] == 1].groupby(c).ngroups,
                         'treated_cells': panel.loc[(panel[f] == 1) & (tc == 1)].groupby(c).ngroups} for f in flags}).T
print(summ.to_string())

panel.to_parquet(out_panel, index=False)
print(f'saved: {out_panel.name} ({len(panel):,} rows, {panel.shape[1]} cols)')
del panel, pat, treat, clong, scope, t; gc.collect()

                 rows  cells  treated_cells
in_pat         153450   3410            290
in_cit         146630   3410            290
in_member      112090   3410            290
left_cens        1632     34             34
few_pre          3744     78             78
multi_ep         1776     37             37
post_ep2          806     37             37
never_exp      149712   3119              0
in_scope        11616   1077            280
scope_only       8622    970            174
scope_ever      51696   1077            280
out_scope         187     41             41
isic_exposed    20498   1423            287
pat_zero_cell    5376    112              0
alp_artefact        0      0              0
treat_2d          849     90             90
saved: panel.parquet (163,680 rows, 34 cols)


0